In [1]:

!pip install pandas openpyxl plotly

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Load your raw Excel dataset
df = pd.read_excel("DAV_Raw_2000_Rows.xlsx")

# Display first five rows
print(df.head())

# Check number of rows and columns
print("Dataset shape:", df.shape)

# Display column names
print(df.columns.tolist())


   ResponseId                                         MainBranch  \
0         128  I am not primarily a developer, but I write co...   
1         314  I am not primarily a developer, but I write co...   
2         324                     I am a developer by profession   
3         352                     I am a developer by profession   
4         374  I am not primarily a developer, but I write co...   

           Employment                   Country       US_State UK_Country  \
0  Student, full-time  United States of America       New York        NaN   
1  Employed full-time                Costa Rica            NaN        NaN   
2  Employed full-time  United States of America  New Hampshire        NaN   
3  Employed full-time                   Germany            NaN        NaN   
4  Employed full-time                    Sweden            NaN        NaN   

                                             EdLevel     Age1stCode  \
0  Secondary school (e.g. American high school, G...   5 

In [2]:

# Make a copy of the raw data
df = df.copy()

# Remove duplicate rows
df = df.drop_duplicates()

# Remove extra spaces from text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Convert coding experience into numeric values
def convert_years(value):
    if pd.isna(value):
        return float("nan")
    value = str(value).strip()
    if value == "Less than 1 year":
        return 0.5
    if value == "More than 50 years":
        return 51
    try:
        return float(value)
    except ValueError:
        return float("nan")

df["YearsCode_num"] = df["YearsCode"].apply(convert_years)
df["YearsCodePro_num"] = df["YearsCodePro"].apply(convert_years)

# Check the cleaned dataset
print("Rows and columns:", df.shape)
print("Missing values:\n", df.isnull().sum().head(10))
print(df.head())


Rows and columns: (2000, 50)
Missing values:
 ResponseId       0
MainBranch       0
Employment       4
Country          0
US_State      1633
UK_Country    1898
EdLevel          2
Age1stCode       3
LearnCode        8
YearsCode       37
dtype: int64
   ResponseId                                         MainBranch  \
0         128  I am not primarily a developer, but I write co...   
1         314  I am not primarily a developer, but I write co...   
2         324                     I am a developer by profession   
3         352                     I am a developer by profession   
4         374  I am not primarily a developer, but I write co...   

           Employment                   Country       US_State UK_Country  \
0  Student, full-time  United States of America       New York        NaN   
1  Employed full-time                Costa Rica            NaN        NaN   
2  Employed full-time  United States of America  New Hampshire        NaN   
3  Employed full-time             

In [3]:

fig1 = px.histogram(
    df.dropna(subset=["YearsCode_num"]),
    x="YearsCode_num",
    nbins=30,
    title="Distribution of Years of Coding Experience"
)

fig1.update_layout(
    xaxis_title="Years of Coding Experience",
    yaxis_title="Number of Developers",
    xaxis_rangeslider_visible=True
)

fig1.show()


In [4]:

fig2 = px.scatter(
    df.dropna(subset=["YearsCode_num", "ConvertedCompYearly"]),
    x="YearsCode_num",
    y="ConvertedCompYearly",
    color="Employment",
    hover_data=["Country", "DevType", "EdLevel"],
    title="Coding Experience vs Yearly Compensation",
    log_y=True
)

fig2.update_layout(
    xaxis_title="Years of Coding Experience",
    yaxis_title="Yearly Compensation (reported converted value)"
)

fig2.show()


In [5]:

fig3 = px.box(
    df.dropna(subset=["Employment", "ConvertedCompYearly"]),
    x="Employment",
    y="ConvertedCompYearly",
    color="Employment",
    points="all",
    title="Yearly Compensation by Employment Type"
)

fig3.update_layout(
    xaxis_title="Employment Type",
    yaxis_title="Yearly Compensation"
)

fig3.show()


In [6]:

df3 = df.dropna(
    subset=[
        "YearsCode_num",
        "YearsCodePro_num",
        "ConvertedCompYearly"
    ]
)

fig4 = px.scatter_3d(
    df3,
    x="YearsCode_num",
    y="YearsCodePro_num",
    z="ConvertedCompYearly",
    color="Employment",
    hover_data=["Country", "DevType"],
    title="3D Analysis of Developer Experience and Compensation"
)

fig4.update_layout(
    scene=dict(
        xaxis_title="Total Coding Experience",
        yaxis_title="Professional Coding Experience",
        zaxis_title="Yearly Compensation"
    )
)

fig4.show()


In [7]:

df5 = df.dropna(
    subset=["YearsCode_num", "ConvertedCompYearly", "Age"]
).copy()

age_order = [
    age for age in [
        "Under 18 years old",
        "18-24 years old",
        "25-34 years old",
        "35-44 years old",
        "45-54 years old",
        "55-64 years old",
        "65 years or older"
    ]
    if age in df5["Age"].unique()
]

fig5 = px.scatter(
    df5,
    x="YearsCode_num",
    y="ConvertedCompYearly",
    animation_frame="Age",
    category_orders={"Age": age_order},
    color="Employment",
    hover_data=["Country", "DevType"],
    range_x=[0, 55],
    range_y=[
        0,
        df5["ConvertedCompYearly"].quantile(0.99)
    ],
    title="Developer Experience and Compensation by Age Group"
)

fig5.update_layout(
    xaxis_title="Years of Coding Experience",
    yaxis_title="Yearly Compensation"
)

fig5.show()

# Animation compares developer groups by age, whereas a static chart
# shows only the selected group or all respondents together.


In [8]:

fig6 = go.Figure()

# Coding experience
fig6.add_trace(go.Scatter(
    x=df["ResponseId"],
    y=df["YearsCode_num"],
    mode="markers",
    name="Total Coding Experience"
))

# Professional coding experience
fig6.add_trace(go.Scatter(
    x=df["ResponseId"],
    y=df["YearsCodePro_num"],
    mode="markers",
    name="Professional Coding Experience",
    visible=False
))

# Yearly compensation
fig6.add_trace(go.Scatter(
    x=df["ResponseId"],
    y=df["ConvertedCompYearly"],
    mode="markers",
    name="Yearly Compensation",
    visible=False
))

fig6.update_layout(
    title="Developer Survey: Select a Metric",
    xaxis_title="Response ID",
    yaxis_title="Years of Coding Experience",
    updatemenus=[dict(
        type="dropdown",
        x=1.0,
        y=1.15,
        buttons=[
            dict(
                label="Coding Experience",
                method="update",
                args=[
                    {"visible": [True, False, False]},
                    {"yaxis": {"title": "Years of Coding Experience"}}
                ]
            ),
            dict(
                label="Professional Experience",
                method="update",
                args=[
                    {"visible": [False, True, False]},
                    {"yaxis": {"title": "Years of Professional Experience"}}
                ]
            ),
            dict(
                label="Yearly Compensation",
                method="update",
                args=[
                    {"visible": [False, False, True]},
                    {"yaxis": {"title": "Yearly Compensation"}}
                ]
            )
        ]
    )]
)

fig6.show()
